In [1]:
# Install required libraries
!pip install -q transformers torch pandas scikit-learn

In [1]:
from google.colab import files

uploaded = files.upload()


KeyboardInterrupt: 

In [1]:
from google.colab import files

uploaded = files.upload()

Saving model.safetensors to model.safetensors
Saving config.json to config.json


In [2]:
import os

print("Files currently in /content:")
print(os.listdir("/content"))

Files currently in /content:
['.config', 'model.safetensors', 'config.json', 'sample_data']


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Original tokenizer used by Legal-RoBERTa
tokenizer = AutoTokenizer.from_pretrained("lexlms/legal-roberta-base")

# Your fine-tuned model
model = AutoModelForSequenceClassification.from_pretrained(
    "/content"
)

print("Model loaded successfully!")
print("Number of labels:", model.config.num_labels)

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully!
Number of labels: 42


In [4]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

Saving test.csv to test.csv


In [5]:
import pandas as pd

df = pd.read_csv("/content/test.csv")

print("Number of test samples:", len(df))
print("Columns:", df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

Number of test samples: 1512
Columns: ['contract', 'clause_text', 'label', 'all_labels']

First 5 rows:


,contract,clause_text,label,all_labels
0,ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-...,"(the ""Parties"" or ""Joint Venturers"" if referre...",Parties,Parties
1,ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-...,All books and records of every kind and charac...,Audit Rights,Audit Rights
2,ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-...,CCGI,Parties,Parties
3,ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-...,"Collectible Concepts Group, Inc.",Parties,Parties
4,ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-...,Division of Income and Losses. All income and ...,Revenue/Profit Sharing,Revenue/Profit Sharing


In [6]:
import torch
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm.auto import tqdm

# Put model in evaluation mode
model.eval()

# Use GPU if Colab provides one
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Using device:", device)

predictions = []
confidences = []

# Predict in batches
batch_size = 16

for i in tqdm(range(0, len(df), batch_size)):
    batch_texts = df["clause_text"].iloc[i:i+batch_size].tolist()

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)

    batch_confidences, batch_predictions = torch.max(probabilities, dim=-1)

    predictions.extend(batch_predictions.cpu().tolist())
    confidences.extend(batch_confidences.cpu().tolist())

print("\nPrediction completed!")
print("Total predictions:", len(predictions))

Using device: cpu


  0%|          | 0/95 [00:00<?, ?it/s]


Prediction completed!
Total predictions: 1512


In [7]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Convert numeric predictions to the actual class names
id2label = model.config.id2label

predicted_labels = [id2label[int(p)] for p in predictions]
true_labels = df["label"].tolist()

# Overall metrics
accuracy = accuracy_score(true_labels, predicted_labels)
macro_f1 = f1_score(true_labels, predicted_labels, average="macro")
weighted_f1 = f1_score(true_labels, predicted_labels, average="weighted")

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

# Per-class report
report = classification_report(
    true_labels,
    predicted_labels,
    labels=list(id2label.values()),
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()

display(report_df)

Accuracy: 0.8578
Macro F1: 0.8088
Weighted F1: 0.8578


,precision,recall,f1-score,support
Affiliate License-Licensee,0.833333,0.500000,0.625000,10.000000
Affiliate License-Licensor,0.555556,0.526316,0.540541,19.000000
Agreement Date,0.428571,0.782609,0.553846,23.000000
Anti-Assignment,0.981481,0.815385,0.890756,65.000000
Audit Rights,0.977778,0.897959,0.936170,49.000000
Cap On Liability,0.867925,0.821429,0.844037,56.000000
Change Of Control,0.720000,0.782609,0.750000,23.000000
Competitive Restriction Exception,0.437500,0.583333,0.500000,12.000000
Covenant Not To Sue,0.894737,1.000000,0.944444,17.000000
Document Name,0.962264,0.980769,0.971429,52.000000


In [8]:
report_df.to_csv(
    "/content/clause_classification_test_report.csv"
)

print("Report saved!")

Report saved!


In [9]:
results_df = df[["contract", "clause_text", "label"]].copy()

results_df["predicted_label"] = predicted_labels
results_df["confidence"] = confidences

errors_df = results_df[
    results_df["label"] != results_df["predicted_label"]
].copy()

print("Total test samples:", len(results_df))
print("Incorrect predictions:", len(errors_df))
print("Correct predictions:", len(results_df) - len(errors_df))

display(errors_df.head(20))

Total test samples: 1512
Incorrect predictions: 215
Correct predictions: 1297


,contract,clause_text,label,predicted_label,confidence
11,ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-...,The Joint Venture shall commence on the 1st of...,Expiration Date,Notice Period To Terminate Renewal,0.942269
19,AMERICANPHYSICIANSCAPITALINC_03_31_2003-EX-10....,Commission will decrease by .5% effective 10/1...,Revenue/Profit Sharing,Price Restrictions,0.989374
21,AMERICANPHYSICIANSCAPITALINC_03_31_2003-EX-10....,In return for the exclusive appointment of Age...,Non-Compete,Exclusivity,0.979337
22,AMERICANPHYSICIANSCAPITALINC_03_31_2003-EX-10....,"In return for this payment, for a two-year per...",No-Solicit Of Customers,Insurance,0.939639
42,ASHWORTHINC_01_29_1999-EX-10.(D)-PROMOTION AGR...,"During the Term, neither Nantz Communications ...",Exclusivity,Non-Compete,0.954242
43,ASHWORTHINC_01_29_1999-EX-10.(D)-PROMOTION AGR...,"Except as otherwise provided herein, and subje...",Exclusivity,License Grant,0.939253
56,ASHWORTHINC_01_29_1999-EX-10.(D)-PROMOTION AGR...,The Company acknowledges that Nantz Communicat...,Most Favored Nation,Competitive Restriction Exception,0.453621
76,AgapeAtpCorp_20191202_10-KA_EX-10.1_11911128_E...,The Manufacturer agrees that the Customer has ...,Non-Compete,Competitive Restriction Exception,0.825375
80,AgapeAtpCorp_20191202_10-KA_EX-10.1_11911128_E...,The Manufacturer covenants not to sell any pro...,Exclusivity,Anti-Assignment,0.611480
81,AgapeAtpCorp_20191202_10-KA_EX-10.1_11911128_E...,The Manufacturer grants exclusive rights to th...,Competitive Restriction Exception,Exclusivity,0.906785


In [10]:
errors_df.to_csv(
    "/content/clause_classification_errors.csv",
    index=False
)

print("Error analysis saved!")

Error analysis saved!


In [11]:
# Create detailed prediction results
results_df = df[["contract", "clause_text", "label"]].copy()

results_df["predicted_label"] = predicted_labels
results_df["confidence"] = confidences

# Keep only incorrect predictions
errors_df = results_df[
    results_df["label"] != results_df["predicted_label"]
].copy()

print("Total test samples:", len(results_df))
print("Incorrect predictions:", len(errors_df))
print("Correct predictions:", len(results_df) - len(errors_df))

# Save error analysis
errors_df.to_csv(
    "/content/clause_classification_errors.csv",
    index=False
)

print("\nError file saved!")

Total test samples: 1512
Incorrect predictions: 215
Correct predictions: 1297

Error file saved!


In [12]:
display(
    errors_df[
        ["clause_text", "label", "predicted_label", "confidence"]
    ].head(20)
)

,clause_text,label,predicted_label,confidence
11,The Joint Venture shall commence on the 1st of...,Expiration Date,Notice Period To Terminate Renewal,0.942269
19,Commission will decrease by .5% effective 10/1...,Revenue/Profit Sharing,Price Restrictions,0.989374
21,In return for the exclusive appointment of Age...,Non-Compete,Exclusivity,0.979337
22,"In return for this payment, for a two-year per...",No-Solicit Of Customers,Insurance,0.939639
42,"During the Term, neither Nantz Communications ...",Exclusivity,Non-Compete,0.954242
43,"Except as otherwise provided herein, and subje...",Exclusivity,License Grant,0.939253
56,The Company acknowledges that Nantz Communicat...,Most Favored Nation,Competitive Restriction Exception,0.453621
76,The Manufacturer agrees that the Customer has ...,Non-Compete,Competitive Restriction Exception,0.825375
80,The Manufacturer covenants not to sell any pro...,Exclusivity,Anti-Assignment,0.611480
81,The Manufacturer grants exclusive rights to th...,Competitive Restriction Exception,Exclusivity,0.906785


In [13]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Confidence threshold for the safety filter
CONFIDENCE_THRESHOLD = 0.60

# Apply confidence-based filtering
filtered_predictions = []

for prediction, confidence in zip(predicted_labels, confidences):
    if confidence < CONFIDENCE_THRESHOLD:
        filtered_predictions.append("LOW_CONFIDENCE")
    else:
        filtered_predictions.append(prediction)

# Number of predictions retained
retained_mask = np.array(confidences) >= CONFIDENCE_THRESHOLD

retained_true = np.array(true_labels)[retained_mask]
retained_pred = np.array(predicted_labels)[retained_mask]

print("Confidence threshold:", CONFIDENCE_THRESHOLD)
print("Total samples:", len(true_labels))
print("Low-confidence samples:", np.sum(~retained_mask))
print("Retained predictions:", np.sum(retained_mask))
print("Coverage:", f"{np.mean(retained_mask) * 100:.2f}%")

# Evaluate only predictions above the confidence threshold
if len(retained_true) > 0:
    filtered_accuracy = accuracy_score(retained_true, retained_pred)
    filtered_macro_f1 = f1_score(
        retained_true,
        retained_pred,
        average="macro"
    )

    print("\nPerformance on retained predictions:")
    print(f"Accuracy: {filtered_accuracy:.4f}")
    print(f"Macro F1: {filtered_macro_f1:.4f}")

Confidence threshold: 0.6
Total samples: 1512
Low-confidence samples: 122
Retained predictions: 1390
Coverage: 91.93%

Performance on retained predictions:
Accuracy: 0.8986
Macro F1: 0.8332


In [14]:
# Save predictions with confidence filtering
filtered_results_df = results_df.copy()

filtered_results_df["filtered_prediction"] = filtered_predictions
filtered_results_df["low_confidence"] = (
    filtered_results_df["confidence"] < CONFIDENCE_THRESHOLD
)

filtered_results_df.to_csv(
    "/content/clause_classification_filtered_results.csv",
    index=False
)

print("Filtered evaluation file saved!")

Filtered evaluation file saved!


In [15]:
summary = f"""Clause Classification Test Evaluation
========================================

Test samples: {len(true_labels)}

Original performance:
Accuracy: {accuracy:.4f}
Macro F1: {macro_f1:.4f}
Weighted F1: {weighted_f1:.4f}

Incorrect predictions: {len(errors_df)}
Correct predictions: {len(results_df) - len(errors_df)}

Confidence-based improvement:
Threshold: {CONFIDENCE_THRESHOLD}
Low-confidence predictions: {np.sum(~retained_mask)}
Retained predictions: {np.sum(retained_mask)}
Coverage: {np.mean(retained_mask) * 100:.2f}%
Filtered Accuracy: {filtered_accuracy:.4f}
Filtered Macro F1: {filtered_macro_f1:.4f}
"""

with open("/content/evaluation_summary.txt", "w") as f:
    f.write(summary)

print(summary)

Clause Classification Test Evaluation

Test samples: 1512

Original performance:
Accuracy: 0.8578
Macro F1: 0.8088
Weighted F1: 0.8578

Incorrect predictions: 215
Correct predictions: 1297

Confidence-based improvement:
Threshold: 0.6
Low-confidence predictions: 122
Retained predictions: 1390
Coverage: 91.93%
Filtered Accuracy: 0.8986
Filtered Macro F1: 0.8332



In [16]:
from google.colab import files

files.download("/content/clause_classification_test_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
files.download("/content/clause_classification_errors.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
files.download("/content/clause_classification_filtered_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
files.download("/content/evaluation_summary.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>